# Lab 7c: Measuring and comparing earthquake magnitudes

> **Colab note:** This notebook is designed to run on **Google Colab**. The first code cell installs dependencies. [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UW-geophysics-edu/ess-412-512-intro2seismology/blob/main/notebooks/07b_magnitudes.ipynb)

This lab operationalizes the concepts in **Module 7c: Magnitude estimates**.

## Learning objectives (same as lecture)
By the end of this lab, students should be able to:

1. Explain magnitude as a **calibrated regression** from seismogram measurements (not a single physical property).
2. State what is measured for **ML, mb, Ms, Md/Mc, Mw** and the typical band/phase and distance regime.
3. Predict when and why **saturation** occurs for amplitude-based magnitudes.
4. Interpret **magnitude disagreements** in catalogs in terms of band limits, depth, path/site, and network calibration.
5. Use magnitude *differences* as signals for **source characterization** (e.g., discrimination, harmonization to Mw).

---

## Pedagogical loop
Each exercise follows:
1) **Predict** (write your expectation)
2) **Do** (run code)
3) **Explain** (interpret what happened)

Time target: 60–90 minutes.


In [ ]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt

from obspy import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.geodetics import gps2dist_azimuth

import pandas as pd

# FDSN client (internet required for real-data parts)
FDSN_PROVIDER = "IRIS"
client = Client(FDSN_PROVIDER)

print("ObsPy client:", FDSN_PROVIDER)

## Shared helper functions

We will use:
- a **toy ML-like calibration** (clearly labeled as toy)
- a **fallback synthetic waveform generator** if FDSN retrieval fails

Remember the lecture’s generic form:

\[
M = \log_{10}(\text{measurement}) + F(\Delta,h) + S(\text{station/site}) + C
\]


In [ ]:
def toy_ml_like(A_meters, R_km, site_factor=1.0, a=1.11, b=0.00189, c=0.0):
    """Toy ML-like mapping.

    Parameters
    ----------
    A_meters : float
        Peak displacement amplitude (meters). (Toy choice.)
    R_km : float
        Hypocentral distance proxy in km (here: station-event distance).
    site_factor : float
        Multiplicative site amplification (toy).
    a, b, c : floats
        Toy regression coefficients for distance correction.

    Notes
    -----
    This is NOT a network-standard ML implementation.
    It is used to test sensitivity to amplitude, distance correction, and site effects.
    """
    A_eff = np.abs(A_meters) * site_factor
    # Convert to mm to mimic classic ML amplitude units (toy)
    A_mm = A_eff * 1e3
    if A_mm <= 0:
        return np.nan
    # Toy distance term: a*log10(R) + b*R + c
    return np.log10(A_mm) + a * np.log10(max(R_km, 1.0)) + b * R_km + c


def synthetic_seismogram(duration_s=60.0, fs=50.0, fc_hz=1.0, noise=0.0):
    """Simple synthetic waveform: damped oscillation with controllable corner-like behavior.

    This is only for conceptual exercises (saturation + band-limited amplitude).
    """
    t = np.arange(0, duration_s, 1/fs)
    # Envelope that decays; fc controls oscillation scale (toy)
    env = np.exp(-t / (duration_s/3))
    x = env * np.sin(2*np.pi*fc_hz*t)
    if noise > 0:
        x = x + noise * np.random.randn(len(t))
    return t, x


def safe_remove_response(st, inv, output="DISP"):
    """Remove instrument response with conservative pre-filter."""
    st2 = st.copy()
    # pre_filt in Hz; broad but avoids amplifying extremes
    pre_filt = (0.005, 0.01, 20.0, 25.0)
    st2.remove_response(inventory=inv, output=output, pre_filt=pre_filt, water_level=60)
    return st2

# Exercise A: One waveform → multiple magnitude ideas

We will:
1) retrieve one earthquake + one station waveform
2) remove instrument response
3) filter to a short-period band (ML/mb-like *idea*)
4) measure peak amplitude in a window
5) compute a **toy ML-like** value and test sensitivity

### Predict
Before running code:
- If you increase the **site amplification** by 2×, how should an amplitude-based magnitude change?
- If you change the **distance correction coefficients**, does the magnitude change systematically with distance?

Write your prediction in 1–2 sentences.

In [ ]:
# 1) Choose a moderate-to-large event and retrieve waveforms
# We pick a recent-ish time window but do not hard-code a specific event ID.
# If FDSN retrieval fails, we fall back to synthetic.

use_synthetic_fallback = False

try:
    t0 = UTCDateTime() - 365 * 24 * 3600  # last year
    t1 = UTCDateTime()
    cat = client.get_events(starttime=t0, endtime=t1, minmagnitude=6.2, limit=10)
    event = cat[0]
    origin = event.preferred_origin() or event.origins[0]
    magnitude = event.preferred_magnitude() or event.magnitudes[0]

    ev_lat = origin.latitude
    ev_lon = origin.longitude
    ev_t = origin.time

    print("Selected event time:", ev_t)
    print("Preferred magnitude:", magnitude.magnitude_type, magnitude.mag)

    # Choose a broad station search around the event
    inv = client.get_stations(
        starttime=ev_t,
        endtime=ev_t + 3600,
        latitude=ev_lat,
        longitude=ev_lon,
        maxradius=10.0,  # degrees
        channel="BH?",
        level="response"
    )

    net = inv[0].code
    sta = inv[0][0].code
    loc = inv[0][0][0].location_code
    cha = inv[0][0][0].code

    st_lat = inv[0][0].latitude
    st_lon = inv[0][0].longitude
    dist_m, az, baz = gps2dist_azimuth(ev_lat, ev_lon, st_lat, st_lon)
    R_km = dist_m / 1000.0

    print(f"Station: {net}.{sta}.{loc}.{cha}")
    print(f"Approx distance: {R_km:.1f} km")

    # Fetch a waveform window (generous)
    st = client.get_waveforms(net, sta, loc, cha, ev_t - 60, ev_t + 600, attach_response=False)

except Exception as e:
    print("FDSN retrieval failed; falling back to synthetic.")
    print("Reason:", repr(e))
    use_synthetic_fallback = True


In [ ]:
if use_synthetic_fallback:
    # Synthetic fallback
    fs = 50.0
    t, x = synthetic_seismogram(duration_s=660, fs=fs, fc_hz=1.0, noise=0.01)
    R_km = 100.0

    plt.figure()
    plt.plot(t, x)
    plt.title("Synthetic fallback waveform (caption: damped oscillation + noise)")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (arbitrary)")
    plt.show()

else:
    # Remove instrument response to displacement
    st_disp = safe_remove_response(st, inv, output="DISP")
    tr = st_disp[0]

    # Basic detrend + taper
    tr = tr.copy()
    tr.detrend("linear")
    tr.taper(0.02)

    # Plot displacement
    t = tr.times()
    x = tr.data

    plt.figure()
    plt.plot(t, x)
    plt.title("Displacement seismogram (caption: response-removed, DISPLACEMENT)")
    plt.xlabel("Time since start (s)")
    plt.ylabel("Displacement (m)")
    plt.show()

### Do: measure a short-period peak amplitude

We mimic an **ML/mb-like idea** by filtering to a short-period band and measuring the peak amplitude in a chosen window.

This is *not* a canonical ML or mb implementation—it's a controlled sandbox to see sensitivity.

### Predict
If you narrow the filter to higher frequencies (e.g., 2–8 Hz), do you expect the peak amplitude to increase or decrease? Why?

In [ ]:
fs = 50.0
if not use_synthetic_fallback:
    fs = tr.stats.sampling_rate

def bandpass_and_peak(x, fs, fmin, fmax, t0_s, t1_s):
    from obspy.signal.filter import bandpass
    y = bandpass(x, freqmin=fmin, freqmax=fmax, df=fs, corners=4, zerophase=True)
    i0 = int(max(0, t0_s * fs))
    i1 = int(min(len(y), t1_s * fs))
    peak = np.max(np.abs(y[i0:i1]))
    return y, peak

# Choose a generous window that should include strong arrivals
t0_win, t1_win = 60, 400

# Two short-period bands for comparison
y1, peak1 = bandpass_and_peak(x, fs, fmin=0.5, fmax=5.0, t0_s=t0_win, t1_s=t1_win)
y2, peak2 = bandpass_and_peak(x, fs, fmin=2.0, fmax=8.0, t0_s=t0_win, t1_s=t1_win)

plt.figure()
tt = np.arange(len(x)) / fs
plt.plot(tt, y1, label="0.5–5 Hz")
plt.plot(tt, y2, label="2–8 Hz")
plt.axvspan(t0_win, t1_win, alpha=0.2, label="measurement window")
plt.title("Filtered waveforms (caption: short-period bands for peak measurement)")
plt.xlabel("Time (s)")
plt.ylabel("Displacement (m) or arbitrary")
plt.legend()
plt.show()

print("Peak amplitude (0.5–5 Hz):", peak1)
print("Peak amplitude (2–8 Hz):", peak2)

### Do: compute a toy ML-like magnitude and test sensitivity

We compute:

\[
M_{\mathrm{toy}} = \log_{10}(A_{\mathrm{mm}}) + a\log_{10}(R) + bR + c
\]

### Predict
1) If site amplification is 2×, how much should magnitude shift (approximately)?
2) If you increase the coefficient multiplying \(\log_{10}(R)\), does that increase or decrease inferred magnitude for distant stations?

In [ ]:
A = peak1  # choose one band as the measurement

M_toy = toy_ml_like(A, R_km, site_factor=1.0)
M_toy_site2 = toy_ml_like(A, R_km, site_factor=2.0)
M_toy_alt = toy_ml_like(A, R_km, site_factor=1.0, a=1.30, b=0.00189, c=0.0)

print(f"Toy ML-like magnitude (site=1): {M_toy:.2f}")
print(f"Toy ML-like magnitude (site=2): {M_toy_site2:.2f}")
print(f"Toy ML-like magnitude (alt distance a=1.30): {M_toy_alt:.2f}")

print("\nDelta(site 2x - site 1x):", M_toy_site2 - M_toy)
print("Delta(alt - base):", M_toy_alt - M_toy)

### Explain (2–3 sentences)
1) Did the site amplification behave like your prediction? Why is this a problem for amplitude-based magnitudes?
2) How did the distance correction change the magnitude? What does that imply about network-to-network differences?

### Check (hint)
- A factor of 10 in amplitude changes \(\log_{10}(A)\) by 1.
- A factor of 2 in amplitude changes \(\log_{10}(A)\) by ~0.301.


# Exercise B: Catalog magnitudes for the same event(s)

Now we use ObsPy to pull **multiple magnitude estimates** for the same earthquakes.

We will:
- fetch a small event set
- extract all magnitude entries per event
- build a table: type, value, agency/author (if available)
- compare pairs like (mb vs Mw) and (ML vs Mw) when both exist

### Predict
1) Do you expect every event to have **Mw** listed? Why or why not?
2) Do you expect **mb** to correlate perfectly with **Mw** at large magnitude? Why?


In [ ]:
# Fetch a handful of events with moderate magnitude
t0 = UTCDateTime() - 365 * 24 * 3600
t1 = UTCDateTime()

cat2 = client.get_events(starttime=t0, endtime=t1, minmagnitude=6.0, limit=8)
print("Events retrieved:", len(cat2))

rows = []
for i, ev in enumerate(cat2):
    origin = ev.preferred_origin() or ev.origins[0]
    evid = str(ev.resource_id)
    for mag in ev.magnitudes:
        rows.append({
            "event_index": i,
            "event_id": evid,
            "origin_time": origin.time.datetime,
            "mag_type": mag.magnitude_type,
            "mag": mag.mag,
            "author": getattr(mag.creation_info, "author", None) if mag.creation_info else None,
            "agency": getattr(mag.creation_info, "agency_id", None) if mag.creation_info else None
        })

df = pd.DataFrame(rows)
df.head(10)

In [ ]:
# Show a compact table per event
for idx in sorted(df["event_index"].unique()):
    sub = df[df["event_index"] == idx].copy()
    print("\n=== Event", idx, "===")
    print("Origin time:", sub["origin_time"].iloc[0])
    display(sub[["mag_type", "mag", "agency", "author"]].sort_values("mag_type"))

### Do: compare magnitude pairs when available

We pivot each event into one row with columns for magnitude types.

### Predict
If you plot **mb vs Mw** for larger events, do you expect the slope to stay ~1? Or to flatten? Explain using the lecture’s saturation idea.

In [ ]:
# Pivot to one row per event with columns for each magnitude type (taking first available entry per type)
df_pivot = (df.sort_values(["event_index", "mag_type"])  # stable
              .groupby(["event_index", "mag_type"])
              .first()
              .reset_index())

wide = df_pivot.pivot(index="event_index", columns="mag_type", values="mag")
wide["origin_time"] = df_pivot.groupby("event_index")["origin_time"].first()
wide

In [ ]:
def scatter_if_present(wide, xcol, ycol):
    if xcol not in wide.columns or ycol not in wide.columns:
        print(f"Missing {xcol} or {ycol} in this sample.")
        return
    sub = wide[[xcol, ycol]].dropna()
    if len(sub) < 3:
        print(f"Not enough points for {xcol} vs {ycol} (need ~3+).")
        return
    plt.figure()
    plt.scatter(sub[xcol], sub[ycol])
    plt.title(f"{ycol} vs {xcol} (caption: catalog magnitudes for same events)")
    plt.xlabel(xcol)
    plt.ylabel(ycol)
    plt.show()

scatter_if_present(wide, "mb", "Mw")
scatter_if_present(wide, "ML", "Mw")
scatter_if_present(wide, "Ms", "Mw")

### Explain (2–3 sentences)
- Which magnitude types appear most often in your sample?
- Do you see hints of saturation or systematic offsets in any pair plot?
- If an agency reports multiple entries for the same type, what might that mean?

### Check (hint)
- Saturation often appears as a **flattening**: increases in Mw do not produce proportional increases in short-period magnitudes.


# Exercise C: Saturation demonstration (synthetic, concept-first)

Goal: **see** why short-period magnitudes saturate.

We use a toy source spectrum idea:
- larger events have larger moment (low-frequency level grows)
- larger events have lower corner frequency (energy shifts to longer periods)

We will compare:
- a **short-period proxy amplitude** at a fixed frequency (band-limited)
- a **long-period proxy** (moment-like)

### Predict
As event size increases and corner frequency decreases:
1) What happens to amplitude at **short period** (high frequency)?
2) What happens to the **long-period level** (moment proxy)?


In [ ]:
def omega_square_displacement_spectrum(f, M0, fc, k=1.0):
    """Toy omega-square displacement spectrum amplitude.

    |U(f)| ~ M0 / (1 + (f/fc)^2)
    This captures: low-f plateau proportional to M0; high-f decay ~ f^-2.
    """
    return k * M0 / (1.0 + (f / fc)**2)

# Define a toy Mw grid and map to toy M0
Mw_vals = np.linspace(3.0, 8.0, 26)
# Toy scaling: log10(M0) ~ 1.5*Mw + const (consistent with Mw definition up to a constant)
log10M0 = 1.5 * Mw_vals + 9.0
M0_vals = 10**log10M0

# Toy corner frequency decreases with Mw (conceptual)
fc_vals = 10**(1.5 - 0.3*(Mw_vals - 3.0))  # Hz

f_short = 1.0   # 1 Hz (short-period proxy)
f_long = 0.02   # 50 s (long-period proxy)

A_short = omega_square_displacement_spectrum(f_short, M0_vals, fc_vals)
A_long = omega_square_displacement_spectrum(f_long, M0_vals, fc_vals)

plt.figure()
plt.plot(Mw_vals, np.log10(A_short), label="log10 short-period proxy (1 Hz)")
plt.plot(Mw_vals, np.log10(A_long), label="log10 long-period proxy (0.02 Hz)")
plt.title("Toy saturation demonstration (caption: band-limited vs long-period scaling)")
plt.xlabel("Mw (toy)")
plt.ylabel("log10 amplitude proxy")
plt.legend()
plt.show()

### Do: turn the short-period proxy into a toy magnitude

Compute a toy magnitude-like number:
\[
M_{\mathrm{short}} = \log_{10}(A_{1\,Hz}) + \text{constant}
\]

### Predict
Should \(M_{\mathrm{short}}\) track Mw linearly across the whole range? Why/why not?

In [ ]:
M_short = np.log10(A_short) - np.mean(np.log10(A_short[:3])) + Mw_vals[0]  # anchored for display
M_long = np.log10(A_long) - np.mean(np.log10(A_long[:3])) + Mw_vals[0]

plt.figure()
plt.plot(Mw_vals, M_short, label="toy short-period magnitude")
plt.plot(Mw_vals, M_long, label="toy long-period magnitude")
plt.plot(Mw_vals, Mw_vals, label="Mw (reference line)")
plt.title("Toy magnitudes (caption: short-period saturates relative to Mw)")
plt.xlabel("Mw (toy)")
plt.ylabel("Magnitude-like value")
plt.legend()
plt.show()

### Explain (2–3 sentences)
- Where does the short-period proxy begin to deviate from Mw?
- Connect this to the lecture’s statement: “large ruptures shift energy to longer periods.”

### Check (hint)
- If corner frequency drops, fixed 1 Hz sits deeper in the high-frequency roll-off, reducing sensitivity to M0 growth.


# Wrap-up: catalog interpretation prompts

Answer briefly (2–3 sentences each):

1) In a hazard model, why is $M_w$ typically preferred over $M_L$ or $m_b$?
2) For rapid monitoring of small events in a local network, why might $M_d/M_c$ be attractive?
3) If two agencies disagree systematically on $M_L$, list two plausible calibration-related reasons.
4) Give one reason magnitude differences can help in discrimination or source characterization.
